<a href="https://colab.research.google.com/github/bora7-glitch/Authentication_App/blob/main/slrnet8_asl_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SLRNet-8: American Sign Language (ASL) Recognition

This notebook implements the **SLRNet-8** Convolutional Neural Network architecture for American Sign Language recognition following the research paper specifications with 10-fold cross-validation.

### Supported Paper Datasets:
1. **Sign Language Digits Dataset** (Auto-downloaded via GitHub/Kaggle)
2. **ASL Alphabet Dataset** (`grassknoted/asl-alphabet`)
3. **Massey University (MU) HandImages ASL**
4. **Surrey ASL Fingerspelling Dataset**

In [ ]:
# Cell 1: Environment Setup and Automatic Dataset Download
import os
import sys
import glob
import shutil
import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras import layers, models, initializers, regularizers, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Automatic download of the Paper's Sign Language Digits Dataset
DATASET_REPO = "https://github.com/ardamavi/Sign-Language-Digits-Dataset.git"
DATASET_DIR = "Sign-Language-Digits-Dataset/Dataset"

if not os.path.exists(DATASET_DIR):
    print("Downloading Sign Language Digits Dataset from GitHub...")
    os.system(f"git clone {DATASET_REPO}")

if os.path.exists(DATASET_DIR):
    print(f"Dataset successfully loaded at: {DATASET_DIR}")
    classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
    print(f"Detected {len(classes)} classes: {classes}")
else:
    print("Using custom/local DATASET_DIR path. Set DATASET_DIR to your dataset directory.")

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

Dataset successfully loaded at: Sign-Language-Digits-Dataset/Dataset
Detected 10 classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
TensorFlow Version: 2.20.0
GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Cell 2: Data Preprocessing Pipeline
def load_dataset(dataset_dir, target_size=(64, 64)):
    """
    Loads raw images, converts them to grayscale, resizes to 64x64,
    and normalizes pixel values to [0, 1].
    """
    images = []
    labels = []

    if not os.path.exists(dataset_dir):
        raise FileNotFoundError(f"Dataset directory '{dataset_dir}' does not exist.")

    class_names = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    print(f"Loading images from {len(class_names)} classes...")
    for class_name, class_idx in class_to_idx.items():
        class_folder = os.path.join(dataset_dir, class_name)
        count = 0
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.PNG'):
            for img_path in glob.glob(os.path.join(class_folder, ext)):
                with Image.open(img_path) as img:
                    img_gray = img.convert('L')
                    img_resized = img_gray.resize(target_size, Image.BILINEAR)
                    img_array = np.array(img_resized, dtype=np.float32) / 255.0
                    images.append(np.expand_dims(img_array, axis=-1))
                    labels.append(class_idx)
                    count += 1
        print(f" - Class '{class_name}': {count} images")

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    print(f"Dataset loaded: X.shape = {X.shape}, y.shape = {y.shape}")
    return X, y, class_to_idx

X, y, class_mapping = load_dataset(DATASET_DIR, target_size=(64, 64))
num_classes = len(class_mapping)

Loading images from 10 classes...
 - Class '0': 205 images
 - Class '1': 206 images
 - Class '2': 206 images
 - Class '3': 206 images
 - Class '4': 207 images
 - Class '5': 207 images
 - Class '6': 207 images
 - Class '7': 206 images
 - Class '8': 208 images
 - Class '9': 204 images
Dataset loaded: X.shape = (2062, 64, 64, 1), y.shape = (2062,)


In [ ]:
# Cell 3: SLRNet-8 Architecture Definition
def build_slrnet8(input_shape=(64, 64, 1), num_classes=10):
    init = initializers.RandomNormal(mean=0.0, stddev=0.01)
    reg = regularizers.l2(1e-6)

    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Layer 1: Conv2D (5x5, 32 filters) -> BN -> ReLU
        layers.Conv2D(32, (5, 5), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),

        # Layer 2: Conv2D (5x5, 64 filters) -> BN -> ReLU
        layers.Conv2D(64, (5, 5), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),

        # Layer 3: Conv2D (3x3, 128 filters) -> BN -> ReLU -> MaxPool (2x2, stride 2)
        layers.Conv2D(128, (3, 3), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Layer 4: Conv2D (3x3, 256 filters) -> BN -> ReLU -> MaxPool (2x2, stride 2)
        layers.Conv2D(256, (3, 3), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Layer 5: Conv2D (3x3, 384 filters) -> BN -> ReLU -> MaxPool (2x2, stride 2)
        layers.Conv2D(384, (3, 3), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Layer 6: Conv2D (3x3, 512 filters) -> BN -> ReLU
        layers.Conv2D(512, (3, 3), padding='same', kernel_initializer=init, kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation('relu'),

        # Pooling & Classification Layers
        layers.GlobalAveragePooling2D(),
        layers.Dense(84, kernel_initializer=init, kernel_regularizer=reg),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax', kernel_initializer=init, kernel_regularizer=reg)
    ])

    return model

sample_model = build_slrnet8(input_shape=(64, 64, 1), num_classes=num_classes)
sample_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 16, 16, 384)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 384)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 512)      │     1,769,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             

 Total params: 3,125,670 (11.92 MB)

 Trainable params: 3,122,918 (11.91 MB)

 Non-trainable params: 2,752 (10.75 KB)

In [ ]:
# Cell 4: 10-Fold Cross-Validation Training Loop
from sklearn.model_selection import StratifiedKFold
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import optimizers, callbacks
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_histories = []
fold_evaluations = []

# Data Augmentation as specified in paper
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    fill_mode='nearest'
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*25} Fold {fold}/10 {'='*25}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    y_train_cat = to_categorical(y_train, num_classes=num_classes)
    y_val_cat = to_categorical(y_val, num_classes=num_classes)

    model = build_slrnet8(input_shape=(64, 64, 1), num_classes=num_classes)

    optimizer = optimizers.Adam(learning_rate=0.001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    lr_scheduler = callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.75,
        patience=6,
        mode='max',
        verbose=1
    )

    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        mode='min',
        restore_best_weights=True,
        verbose=1
    )

    train_generator = datagen.flow(X_train, y_train_cat, batch_size=128)

    history = model.fit(
        train_generator,
        steps_per_epoch=64,
        epochs=20,
        validation_data=(X_val, y_val_cat),
        callbacks=[lr_scheduler, early_stopping],
        verbose=1
    )

    eval_res = model.evaluate(X_val, y_val_cat, verbose=0)
    fold_evaluations.append(eval_res)
    fold_histories.append(history.history)
    print(f"Fold {fold} Finished -> Val Loss: {eval_res[0]:.4f} | Val Accuracy: {eval_res[1]:.4f}")


========================= Fold 1/10 =========================
Epoch 1/20
15/64 ━━━━━━━━━━━━━━━━━━━━ 31s 642ms/step - accuracy: 0.1249 - loss: 2.2934

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


64/64 ━━━━━━━━━━━━━━━━━━━━ 22s 184ms/step - accuracy: 0.1558 - loss: 2.2706 - val_accuracy: 0.1014 - val_loss: 2.3037 - learning_rate: 0.0010
Epoch 2/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 46ms/step - accuracy: 0.1925 - loss: 2.0113 - val_accuracy: 0.1014 - val_loss: 2.3077 - learning_rate: 0.0010
Epoch 3/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.2485 - loss: 1.7739 - val_accuracy: 0.1014 - val_loss: 2.6248 - learning_rate: 0.0010
Epoch 4/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - accuracy: 0.3671 - loss: 1.6006 - val_accuracy: 0.0966 - val_loss: 3.1719 - learning_rate: 0.0010
Epoch 5/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.4512 - loss: 1.3745 - val_accuracy: 0.1014 - val_loss: 2.3525 - learning_rate: 0.0010
Epoch 6/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.5105 - loss: 1.1877 - val_accuracy: 0.1014 - val_loss: 4.2939 - learning_rate: 0.0010
Epoch 7/20
15/64 ━━━━━━━━━━━━━━━━━━━━ 10s 205ms/step - accuracy: 0.5460 - loss: 1.1147
Epoch 7: ReduceL

In [ ]:
# Cell 5: Final Cross-Validation Evaluation Summary
accuracies = [res[1] for res in fold_evaluations]
losses = [res[0] for res in fold_evaluations]

print(f"\n{'='*25} 10-Fold Cross-Validation Summary {'='*25}")
for i, (loss, acc) in enumerate(zip(losses, accuracies), 1):
    print(f"Fold {i:02d}: Accuracy = {acc * 100:.2f}%, Loss = {loss:.4f}")

print("-" * 50)
print(f"Mean Validation Accuracy: {np.mean(accuracies) * 100:.2f}% (+/- {np.std(accuracies) * 100:.2f}%)")
print(f"Mean Validation Loss:     {np.mean(losses):.4f} (+/- {np.std(losses):.4f})")


========================= 10-Fold Cross-Validation Summary =========================
Fold 01: Accuracy = 10.14%, Loss = 2.3037
Fold 02: Accuracy = 11.11%, Loss = 2.3018
Fold 03: Accuracy = 9.71%, Loss = 2.3033
Fold 04: Accuracy = 10.19%, Loss = 2.3066
Fold 05: Accuracy = 9.71%, Loss = 2.3031
Fold 06: Accuracy = 10.19%, Loss = 2.3176
Fold 07: Accuracy = 10.19%, Loss = 2.3035
Fold 08: Accuracy = 9.71%, Loss = 2.3066
Fold 09: Accuracy = 9.71%, Loss = 2.3049
Fold 10: Accuracy = 10.19%, Loss = 2.3076
--------------------------------------------------
Mean Validation Accuracy: 10.09% (+/- 0.41%)
Mean Validation Loss:     2.3059 (+/- 0.0043)
